# Incremental Classifier Exploration

This notebook trains the same River model used by the API. It expects the ten intake features and twelve statutory categories configured by the front end, and persists the artifact consumed by `classifier.py`.

Bootstrap training uses `Temp/rag_router_dataset_v1_10000.csv`; the differently shaped 100k product/regulatory dataset is not compatible. A selected retrieval category is used as one weak incremental label only when a question is sent with that changed category. It updates future predictions, not the current assessment.

In [5]:
FEATURES = [
    "subject_matter",
    "primary_objective",
    "technical_invention",
    "brand_identifier",
    "product_appearance",
    "geographical_origin",
    "creative_expression",
    "bio_or_plant_matter",
    "confidentiality",
    "regulated_product_type",
]

CATEGORIES = [
    "Ayurveda-Aahar",
    "Biological Diversity",
    "Copyright",
    "Design",
    "Drug Regulation",
    "Food Regulation",
    "Geographical Indication",
    "Patent",
    "Plant Variety Protection",
    "Regulatory",
    "Trade Secret",
    "Trademark",
]

assert len(FEATURES) == 10
assert len(CATEGORIES) == 12

In [6]:
import river

In [ ]:
print(f"river=={river.__version__}")

In [2]:
from pathlib import Path
import sys
import pandas as pd

workspace = Path.cwd()
backend_dir = next(
    path
    for path in (workspace, workspace / "backend", workspace.parent / "backend")
    if (path / "classifier" / "classifier.py").is_file()
)
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from classifier import IPShaktiClassifier

dataset_path = backend_dir / "notebook" / "Temp" / "rag_router_dataset_v1_10000.csv"
model_path = backend_dir / "models" / "IPShaktiRiverClassifier.pkl"
classifier = IPShaktiClassifier.train_from_csv(dataset_path, model_path)

sample = pd.read_csv(dataset_path).iloc[0]
features = {
    feature: None if pd.isna(sample[feature]) else sample[feature]
    for feature in IPShaktiClassifier.FEATURES
}
classifier.predict(features)

StopIteration: 